In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

path1 = Path('/Users/wehadgoodtimes/.cache/kagglehub/datasets/' \
'nickpudovkin/avito-cup/versions/1/competition/competition/train/impressions/' \
'part-00000-e3428b2d-e8b5-4109-8f1d-0bb6189ed749-c000.zstd.parquet')

root = path1.parents[2]


In [37]:
NUM_COLS = ["f1", "f2", "f3", "f4", "f5"]
COLS = ["item_id", "category_id", *NUM_COLS, "target"]

features = ["f1", "f2", "f3", "f4", "f5", "item_id", "category_id"]
explode_features = [*features, "target"]
target = "target"

train = pd.read_parquet(
    path1,
    columns=["feed_id", *COLS],
)

train = train.explode(
    explode_features,
    ignore_index=True,
)

train["target"] = train["target"].astype("int8")

In [13]:
train

,feed_id,item_id,category_id,f1,f2,f3,f4,f5,target
0,2304337,73328213,"[3, 13, 11, 7, 13, 13, 13, 13, 13, 11, 11, 13,...",0.283893,0.093346,0.238096,0.645679,0.675874,0
1,2304337,47564870,"[3, 13, 11, 7, 13, 13, 13, 13, 13, 11, 11, 13,...",0.736484,0.407478,0.708231,0.940511,0.13994,0
2,2304337,92744465,"[3, 13, 11, 7, 13, 13, 13, 13, 13, 11, 11, 13,...",0.33947,0.314412,0.662287,0.565259,0.653355,0
3,2304337,69672495,"[3, 13, 11, 7, 13, 13, 13, 13, 13, 11, 11, 13,...",0.017857,0.0,0.065683,0.36191,0.529142,1
4,2304337,92017650,"[3, 13, 11, 7, 13, 13, 13, 13, 13, 11, 11, 13,...",0.633698,0.790085,0.708231,0.233312,0.790685,0
...,...,...,...,...,...,...,...,...,...
476296,1291292,12250727,"[7, 7, 50, 48, 7, 27, 7, 48, 48, 7, 53, 48, 7]",0.473606,0.525216,0.562355,0.947137,0.016129,0
476297,1291292,105578074,"[7, 7, 50, 48, 7, 27, 7, 48, 48, 7, 53, 48, 7]",0.355702,0.476639,0.712392,0.676005,0.34033,1
476298,1291292,102251541,"[7, 7, 50, 48, 7, 27, 7, 48, 48, 7, 53, 48, 7]",0.017857,0.370656,0.016129,0.104892,0.030033,0
476299,1291292,93941939,"[7, 7, 50, 48, 7, 27, 7, 48, 48, 7, 53, 48, 7]",0.519322,0.525216,0.562355,0.127863,0.016129,1


# MODEL

In [28]:
from catboost import CatBoostRanker, Pool
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import ndcg_score
from tqdm.auto import tqdm

0.001

In [40]:
pool = Pool(
    data=train[features],
    label=train["target"],
    group_id=train["feed_id"],
    cat_features=train["category_id"],
)

model_prototype = CatBoostRanker(
    iterations=1000,
    learning_rate=1e-3,
    depth=5,
    loss_function="YetiRank",
    eval_metric="NDCG:type=Base",
    random_seed=42,
    allow_writing_files=False,
    verbose=100,
)

CatBoostError: feature names should be a sequence, but got 0          3
1         13
2         11
3          7
4         13
          ..
476296    48
476297     7
476298    53
476299    48
476300     7
Name: category_id, Length: 476301, dtype: object

In [ ]:
scores = []

skfold = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for train_idx, val_idx in tqdm(skfold.split(train[features], train["target"], groups=train["feed_id"])):
    train_pool = pool.slice(train_idx)
    val_pool = pool.slice(val_idx)

    model = clone(model_prototype)
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=100,
    )

    score = model.eval_metrics(
        val_pool,
        ["NDCG"],
    )["NDCG:type=Base"][-1]

    scores.append(score)

np.mean(scores), np.std(scores)

0it [00:00, ?it/s]

0:	test: 0.3003893	best: 0.3003893 (0)	total: 101ms	remaining: 1m 41s
100:	test: 0.3305550	best: 0.3305550 (100)	total: 9.9s	remaining: 1m 28s
200:	test: 0.3323164	best: 0.3323164 (200)	total: 20.3s	remaining: 1m 20s
300:	test: 0.3351657	best: 0.3354738 (297)	total: 30.5s	remaining: 1m 10s
400:	test: 0.3537483	best: 0.3542672 (381)	total: 40.6s	remaining: 1m
500:	test: 0.3580294	best: 0.3582626 (485)	total: 50.5s	remaining: 50.3s
600:	test: 0.3596399	best: 0.3600358 (596)	total: 1m	remaining: 39.9s


In [ ]:
train

,feed_id,item_id,category_id,f1,f2,f3,f4,f5,target
0,2304337,73328213,3,0.283893,0.093346,0.238096,0.645679,0.675874,0
1,2304337,47564870,13,0.736484,0.407478,0.708231,0.940511,0.13994,0
2,2304337,92744465,11,0.33947,0.314412,0.662287,0.565259,0.653355,0
3,2304337,69672495,7,0.017857,0.0,0.065683,0.36191,0.529142,1
4,2304337,92017650,13,0.633698,0.790085,0.708231,0.233312,0.790685,0
...,...,...,...,...,...,...,...,...,...
476296,1291292,12250727,48,0.473606,0.525216,0.562355,0.947137,0.016129,0
476297,1291292,105578074,7,0.355702,0.476639,0.712392,0.676005,0.34033,1
476298,1291292,102251541,53,0.017857,0.370656,0.016129,0.104892,0.030033,0
476299,1291292,93941939,48,0.519322,0.525216,0.562355,0.127863,0.016129,1
